# BO Online Campaign (Server Queue)

This notebook runs **one seed per execution** for an online BO campaign.

Key behavior:
- Uses queue-backed real simulator with cache-first lookup.
- Persists per-seed artifacts (`json/csv/checkpoint`) under `experiments/bo_online/`.
- Keeps a campaign summary CSV and auto-selects the next pending seed.
- Safe to pause, switch experiments, and resume later.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'src').exists() else NOTEBOOK_DIR.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from optimization import (
    ExperimentRecorder,
    GPSurrogate,
    OnlineBOOptimizer,
    ServerQueueSimulator,
    UCBAcquisition,
    build_bclf_objective,
)
from simulation.result_store import BandpassFilterNamingStrategy, FilesystemResultStore

print(f"Repo root: {REPO_ROOT}")


Repo root: /home/zhanghf/projects/vector


## Campaign Config


In [ ]:
# Queue and output paths
QUEUE_ROOT = Path('queue')
OUTPUT_DIR = Path('experiments/bo_online')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Cache directory for touchstone DB
CACHE_DIR = Path('data/bandpass_coupled_line_filter_export/ext')

# Campaign controls
SEEDS = [0, 1, 2, 3, 4]
FORCE_SEED = None  # e.g., set to 3 to force seed=3 this run
SUMMARY_CSV_PATH = OUTPUT_DIR / 'campaign_summary.csv'

# BO settings
POOL_SIZE = 80
N_INIT = 2
BUDGET = 12
RETRAIN_EVERY = 1
UCB_BETA = 4.0

# Queue wait settings
POLL_INTERVAL_SEC = 2.0
JOB_TIMEOUT_SEC = 60 * 60 * 24

# Template routing
TEMPLATE_NAME = 'bclf_template_v1'
DESIGN_NAME = 'HFSSDesign1'
SETUP_NAME = 'Setup1'
SWEEP_NAME = 'Sweep'

# Geometry definition
VARIABLE_ORDER = ['L', 'L1', 'L2', 'W', 'W1', 'W2', 'S1', 'S2']
VARIABLE_BOUNDS = {
    'L': (7.7, 7.9),
    'L1': (8.0, 8.2),
    'L2': (7.8, 8.0),
    'W': (0.9, 1.1),
    'W1': (0.7, 0.9),
    'W2': (0.9, 1.1),
    'S1': (0.13, 0.16),
    'S2': (0.55, 0.60),
}
UNIT_MAP = {name: 'mm' for name in VARIABLE_ORDER}

objective = build_bclf_objective(
    passband_hz=(5.2e9, 5.9e9),
    lower_stopband_hz=(3.6e9, 4.8e9),
    upper_stopband_hz=(6.5e9, 8.7e9),
    target_center_hz=5.55e9,
    min_passband_s21_db=-5.0,
    max_passband_s11_db=-10.0,
    max_stopband_s21_db=-25.0,
    center_weight=2.0,
    penalty_weight=100.0,
)


## Helpers


In [3]:
def run_id_for_seed(seed: int) -> str:
    return f'bo_online_seed{seed}'


def checkpoint_for_seed(seed: int) -> Path:
    return OUTPUT_DIR / f'{run_id_for_seed(seed)}.checkpoint.json'


def load_campaign_summary(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        if 'seed' in df.columns:
            df = df.sort_values('seed').reset_index(drop=True)
        return df
    return pd.DataFrame(columns=[
        'seed', 'run_id', 'status', 'num_trials', 'best_objective',
        'total_engine_sec', 'total_sim_sec', 'cache_hit_ratio', 'updated_at'
    ])


def upsert_campaign_row(path: Path, row: dict) -> pd.DataFrame:
    df = load_campaign_summary(path)
    seed = int(row['seed'])
    if 'seed' in df.columns and (df['seed'] == seed).any():
        df.loc[df['seed'] == seed, list(row.keys())] = list(row.values())
    else:
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df = df.sort_values('seed').reset_index(drop=True)
    df.to_csv(path, index=False)
    return df


def build_online_components(run_id: str):
    result_store = FilesystemResultStore(
        root_dir=CACHE_DIR,
        naming_strategy=BandpassFilterNamingStrategy(),
        port_num=2,
    )
    simulator = ServerQueueSimulator(
        queue_root=str(QUEUE_ROOT),
        variable_order=VARIABLE_ORDER,
        unit_map=UNIT_MAP,
        template_name=TEMPLATE_NAME,
        design_name=DESIGN_NAME,
        setup_name=SETUP_NAME,
        sweep_name=SWEEP_NAME,
        metrics_to_extract=['max_s11_db', 'min_s21_db'],
        port_num=2,
        result_store=result_store,
        poll_interval_sec=POLL_INTERVAL_SEC,
        job_timeout_sec=JOB_TIMEOUT_SEC,
        run_id=run_id,
        mm_resolution=0.01,
    )
    return simulator


## Campaign Progress


In [4]:
campaign_df = load_campaign_summary(SUMMARY_CSV_PATH)
completed_seeds = set(campaign_df.loc[campaign_df['status'] == 'completed', 'seed'].astype(int).tolist()) if len(campaign_df) else set()
pending_seeds = [s for s in SEEDS if s not in completed_seeds]
next_seed = pending_seeds[0] if pending_seeds else None

if FORCE_SEED is not None:
    selected_seed = int(FORCE_SEED)
else:
    selected_seed = next_seed

print('completed_seeds =', sorted(completed_seeds))
print('pending_seeds   =', pending_seeds)
print('next_seed       =', next_seed)
print('selected_seed   =', selected_seed)

campaign_df


completed_seeds = []
pending_seeds   = [0, 1, 2, 3, 4]
next_seed       = 0
selected_seed   = 0


,seed,run_id,status,num_trials,best_objective,total_engine_sec,total_sim_sec,cache_hit_ratio,updated_at


## Run One Campaign Step

This cell executes exactly one seed (or skips if campaign already complete).


In [5]:
run_result = None
history_df = None

if selected_seed is None:
    print('All campaign seeds are completed. Skipping optimizer execution.')
else:
    seed = int(selected_seed)
    run_id = run_id_for_seed(seed)
    checkpoint_path = checkpoint_for_seed(seed)

    simulator = build_online_components(run_id)
    optimizer = OnlineBOOptimizer(
        variable_order=VARIABLE_ORDER,
        variable_bounds=VARIABLE_BOUNDS,
        pool_size=POOL_SIZE,
        simulator=simulator,
        objective=objective,
        surrogate=GPSurrogate(length_scale=1.0, noise_variance=1e-8),
        acquisition=UCBAcquisition(beta=UCB_BETA),
        recorder=ExperimentRecorder(output_dir=OUTPUT_DIR),
        budget=BUDGET,
        retrain_every=RETRAIN_EVERY,
        n_init=N_INIT,
        seed=seed,
        run_id=run_id,
        checkpoint_path=checkpoint_path,
        resume=True,
    )

    run_result = optimizer.run()
    history_df = pd.DataFrame(run_result.history)

    total_engine_sec = float(history_df['t_engine_cum_sec'].iloc[-1]) if len(history_df) else 0.0
    total_sim_sec = float(history_df['t_simulator_cum_sec'].iloc[-1]) if len(history_df) else 0.0
    cache_hit_ratio = float(history_df['cache_hit'].fillna(False).astype(bool).mean()) if ('cache_hit' in history_df and len(history_df)) else 0.0

    row = {
        'seed': seed,
        'run_id': run_id,
        'status': 'completed',
        'num_trials': len(history_df),
        'best_objective': float(run_result.state.best_objective),
        'total_engine_sec': total_engine_sec,
        'total_sim_sec': total_sim_sec,
        'cache_hit_ratio': cache_hit_ratio,
        'updated_at': datetime.now(timezone.utc).isoformat(),
    }

    campaign_df = upsert_campaign_row(SUMMARY_CSV_PATH, row)

    print('run_id        =', run_result.run_id)
    print('best_objective=', run_result.state.best_objective)
    print('json          =', run_result.recorder_output.json_path)
    print('csv           =', run_result.recorder_output.csv_path)
    print('checkpoint    =', run_result.checkpoint_path)


KeyboardInterrupt: 

## Campaign-Wide Aggregation


In [ ]:
def load_completed_histories(summary_df: pd.DataFrame) -> pd.DataFrame:
    if len(summary_df) == 0:
        return pd.DataFrame()
    rows = []
    for _, r in summary_df.iterrows():
        if str(r.get('status', '')) != 'completed':
            continue
        run_id = str(r['run_id'])
        csv_path = OUTPUT_DIR / f'{run_id}.csv'
        if not csv_path.exists():
            continue
        h = pd.read_csv(csv_path)
        h['seed'] = int(r['seed'])
        rows.append(h)
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

all_hist = load_completed_histories(campaign_df)
print('completed runs in summary =', len(campaign_df[campaign_df['status'] == 'completed']) if len(campaign_df) else 0)
print('loaded history rows       =', len(all_hist))


In [ ]:
if len(all_hist) == 0:
    print('No completed run history found yet.')
else:
    agg = all_hist.groupby('iteration').agg(
        objective_mean=('objective', 'mean'),
        objective_min=('objective', 'min'),
        objective_max=('objective', 'max'),
        best_mean=('best_so_far', 'mean'),
        best_min=('best_so_far', 'min'),
        best_max=('best_so_far', 'max'),
        engine_mean=('t_engine_cum_sec', 'mean'),
        sim_mean=('t_simulator_cum_sec', 'mean'),
    ).reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(agg['iteration'], agg['objective_mean'], color='tab:blue', label='mean objective')
    axes[0].fill_between(agg['iteration'], agg['objective_min'], agg['objective_max'], color='tab:blue', alpha=0.2, label='min-max range')
    axes[0].set_title('Objective by Iteration')
    axes[0].set_xlabel('iteration')
    axes[0].set_ylabel('objective')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(agg['iteration'], agg['best_mean'], color='tab:green', label='mean best-so-far')
    axes[1].fill_between(agg['iteration'], agg['best_min'], agg['best_max'], color='tab:green', alpha=0.2, label='min-max range')
    axes[1].set_title('Best-So-Far')
    axes[1].set_xlabel('iteration')
    axes[1].set_ylabel('best objective')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(agg['iteration'], agg['engine_mean'], label='engine cum time')
    axes[2].plot(agg['iteration'], agg['sim_mean'], label='simulator cum time')
    axes[2].set_title('Cumulative Time')
    axes[2].set_xlabel('iteration')
    axes[2].set_ylabel('seconds')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.show()


## Reproducibility Table


In [ ]:
campaign_df
